# <font color="#418FDE" size="6.5" uppercase>**Gaussian Processes**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit Gaussian Process Regression and interpret predictive mean and uncertainty. 
- Configure and compose Gaussian process kernels for practical modeling. 
- Apply Gaussian Process Classification while recognizing computational limits. 


## **1. GPR Basics**

### **1.1. GP Distributions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_01_01.jpg?v=1783966259" width="250">



>* Models many possible functions, not one curve
>* Updates beliefs and shows prediction uncertainty

>* Function values are jointly Gaussian
>* Kernels define similarity and shared patterns

>* Predict mean and uncertainty for each input
>* Uncertainty grows with sparse or noisy data



### **1.2. Fitting GPR Models**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_01_02.jpg?v=1783966261" width="250">



>* Model many possible functions, not one curve
>* Combine data and assumptions with uncertainty

>* Similar training points guide predictions most
>* Uncertainty grows with gaps or disagreement

>* Use mean as the best estimate
>* Use uncertainty to judge risk



### **1.3. Alpha Noise**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_01_03.jpg?v=1783966263" width="250">



>* Alpha noise models imperfect training observations
>* Higher noise smooths over individual data points

>* Low alpha can overtrust training points
>* Higher alpha smooths predictions and uncertainty

>* Balance alpha to avoid overfitting or underfitting
>* Alpha improves stability and reflects data trust



In [ ]:
#@title Python Code - Alpha Noise

# Alpha noise controls data trust.
# Small examples reveal smoothing behavior.
# Predictive uncertainty changes near observations.

import numpy as np
import matplotlib.pyplot as plt

# Create reproducible noisy training observations.
rng = np.random.default_rng(7)

# Use tiny one dimensional inputs.
X_train = np.linspace(0, 10, 14)

# Simulate a smooth hidden signal.
y_true = np.sin(X_train) + 0.15 * X_train

# Add realistic measurement noise.
y_train = y_true + rng.normal(0, 0.35, X_train.size)

# Validate the training vector sizes.
assert X_train.shape == y_train.shape

# Create prediction locations for plotting.
X_test = np.linspace(0, 10, 200)

# Define a simple radial basis kernel.
def rbf_kernel(a, b, length_scale, variance):
    a = np.asarray(a).reshape(-1, 1)
    b = np.asarray(b).reshape(1, -1)
    return variance * np.exp(-0.5 * ((a - b) / length_scale) ** 2)

# Compute Gaussian process predictions manually.
def gpr_predict(alpha_noise):
    K = rbf_kernel(X_train, X_train, 1.2, 1.0)
    K = K + alpha_noise * np.eye(X_train.size)
    Ks = rbf_kernel(X_train, X_test, 1.2, 1.0)
    Kss = rbf_kernel(X_test, X_test, 1.2, 1.0)

    # Solve linear systems without matrix inversion.
    weights = np.linalg.solve(K, y_train)
    mean = Ks.T @ weights
    solved = np.linalg.solve(K, Ks)
    cov = Kss - Ks.T @ solved

    # Keep variances nonnegative after rounding.
    std = np.sqrt(np.maximum(np.diag(cov), 0.0))
    return mean, std

# Compare low and higher alpha noise.
mean_low, std_low = gpr_predict(1e-6)
mean_high, std_high = gpr_predict(0.25)

# Print a compact teaching summary.
print("Alpha noise example: manual Gaussian Process Regression")
print("Low alpha trusts points strongly and fits more tightly.")
print("Higher alpha smooths observations and admits measurement noise.")
print(f"Average uncertainty, low alpha: {std_low.mean():.3f}")
print(f"Average uncertainty, higher alpha: {std_high.mean():.3f}")

# Plot predictive means and uncertainty bands.
plt.figure(figsize=(9, 5))
plt.scatter(X_train, y_train, color="black", label="noisy observations")
plt.plot(X_test, mean_low, color="tab:blue", label="alpha = 0.000001")
plt.fill_between(X_test, mean_low - 2 * std_low, mean_low + 2 * std_low, color="tab:blue", alpha=0.15)
plt.plot(X_test, mean_high, color="tab:orange", label="alpha = 0.25")
plt.fill_between(X_test, mean_high - 2 * std_high, mean_high + 2 * std_high, color="tab:orange", alpha=0.15)
plt.title("Alpha noise changes smoothness and uncertainty")
plt.xlabel("Input distance in meters")
plt.ylabel("Measured response")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



## **2. Kernel Design**

### **2.1. Core GP Kernels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_02_01.jpg?v=1783966275" width="250">



>* Kernels define similarity between input points
>* Kernel choice encodes expected function behavior

>* Squared exponential kernels model very smooth changes
>* Matérn kernels handle rougher spatial variation

>* Linear, periodic, noise kernels model distinct patterns
>* Match kernels to system behavior and assumptions



### **2.2. Combining Kernels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_02_02.jpg?v=1783966280" width="250">



>* Combine kernels to model multiple data patterns
>* Richer assumptions improve accuracy and interpretability

>* Add kernels to model separate signal parts
>* Multiply kernels to model interacting patterns

>* Start simple; add kernels only when justified
>* Balance accuracy, uncertainty, interpretability, and generalization



In [ ]:
#@title Python Code - Combining Kernels

# Combine kernels to model richer patterns.
# This example avoids machine learning libraries.
# We visualize covariance from kernel arithmetic.

import numpy as np
import matplotlib.pyplot as plt

# Create a small one dimensional input grid.
x_values = np.linspace(0.0, 10.0, 120)
X = x_values.reshape(-1, 1)

# Validate the grid before computing distances.
assert X.ndim == 2 and X.shape[1] == 1
assert X.shape[0] <= 200 and np.isfinite(X).all()

# Compute pairwise distances in input units.
diff = X - X.T
sqdist = diff ** 2

# Define a smooth trend kernel component.
trend_scale = 4.0
trend_kernel = np.exp(-0.5 * sqdist / trend_scale**2)

# Define a repeating seasonal kernel component.
period = 2.0
season_kernel = np.exp(-2.0 * np.sin(np.pi * diff / period)**2)

# Define a short range irregular kernel component.
local_scale = 0.45
local_kernel = 0.25 * np.exp(-0.5 * sqdist / local_scale**2)

# Add kernels to represent independent signal parts.
added_kernel = trend_kernel + season_kernel + local_kernel

# Multiply kernels to create changing seasonal behavior.
interaction_kernel = trend_kernel * season_kernel

# Select one reference point for covariance curves.
reference_index = 60
reference_x = x_values[reference_index]

# Extract covariance with the reference point.
added_curve = added_kernel[reference_index]
interaction_curve = interaction_kernel[reference_index]

# Print a compact interpretation for learners.
print("Added kernels: trend + season + local variation.")
print("Multiplied kernels: season strength changes smoothly.")
print(f"Reference input: {reference_x:.2f} meters.")

# Plot both combined covariance patterns.
plt.figure(figsize=(8, 4))
plt.plot(x_values, added_curve, label="Added kernels")
plt.plot(x_values, interaction_curve, label="Multiplied kernels")
plt.axvline(reference_x, color="gray", linestyle="--", linewidth=1)
plt.title("Combining Gaussian Process Kernels")
plt.xlabel("Input location, meters")
plt.ylabel("Covariance with reference point")
plt.legend()
plt.tight_layout()
plt.show()



### **2.3. Tuning Kernel Parameters**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_02_03.jpg?v=1783966277" width="250">



>* Adjust kernels to match data patterns
>* Tune length scale, variance, and noise

>* Balance data fit with model simplicity
>* Separate real patterns from noisy fluctuations

>* Validate parameters with real-world knowledge
>* Ensure composed kernels explain distinct patterns



In [ ]:
#@title Python Code - Tuning Kernel Parameters

# Demonstrate Gaussian process kernel tuning.
# Compare smoothness and noise assumptions.
# Use tiny data and one plot.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for reproducibility.
rng = np.random.default_rng(7)

# Create small one dimensional training data.
X_train = np.linspace(0, 10, 14)[:, None]
y_true = np.sin(X_train[:, 0])

# Add modest measurement noise to observations.
y_train = y_true + rng.normal(0, 0.18, size=14)

# Create prediction locations for smooth curves.
X_test = np.linspace(0, 10, 160)[:, None]

# Validate shapes before matrix calculations.
assert X_train.shape == (14, 1)
assert X_test.shape == (160, 1)

# Define a squared exponential kernel.
def rbf_kernel(A, B, length_scale, signal_variance):
    diff = A[:, None, :] - B[None, :, :]
    sqdist = np.sum(diff * diff, axis=2)
    return signal_variance * np.exp(-0.5 * sqdist / length_scale**2)

# Compute Gaussian process posterior predictions.
def gp_predict(length_scale, signal_variance, noise_variance):
    K = rbf_kernel(X_train, X_train, length_scale, signal_variance)
    K = K + noise_variance * np.eye(len(X_train))
    Ks = rbf_kernel(X_train, X_test, length_scale, signal_variance)
    Kss = rbf_kernel(X_test, X_test, length_scale, signal_variance)

    alpha = np.linalg.solve(K, y_train)
    mean = Ks.T @ alpha
    v = np.linalg.solve(K, Ks)
    cov = Kss - Ks.T @ v
    std = np.sqrt(np.maximum(np.diag(cov), 0.0))
    return mean, std

# Compare short and long length scales.
short_mean, short_std = gp_predict(0.45, 1.0, 0.04)
long_mean, long_std = gp_predict(2.20, 1.0, 0.04)

# Print concise interpretation lines.
print("Short length scale: flexible, local, higher wiggle risk.")
print("Long length scale: smoother, broader, may miss local changes.")
print("Noise variance separates sensor error from function uncertainty.")

# Plot predictive means and uncertainty bands.
plt.figure(figsize=(9, 5))
plt.scatter(X_train[:, 0], y_train, color="black", label="observations")
plt.plot(X_test[:, 0], short_mean, color="tab:blue", label="short length scale")
plt.fill_between(X_test[:, 0], short_mean - 2 * short_std, short_mean + 2 * short_std, color="tab:blue", alpha=0.15)

plt.plot(X_test[:, 0], long_mean, color="tab:orange", label="long length scale")
plt.fill_between(X_test[:, 0], long_mean - 2 * long_std, long_mean + 2 * long_std, color="tab:orange", alpha=0.15)
plt.xlabel("distance along road, kilometers")
plt.ylabel("traffic speed change, scaled units")
plt.title("Tuning Gaussian Process Kernel Parameters")

plt.legend(loc="best")
plt.tight_layout()
plt.show()



## **3. GPC Practice**

### **3.1. GP Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_03_01.jpg?v=1783966266" width="250">



>* Predicts class probabilities using flexible functions
>* Models smooth boundaries and uncertainty

>* Learns labels to predict class probabilities
>* Probabilities guide decisions, not guarantees

>* Uncertainty reflects nearby training data patterns
>* Kernel, labels, and scaling affect reliability



### **3.2. Classification Uncertainty Maps**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_03_02.jpg?v=1783966268" width="250">



>* Maps show predicted classes and confidence
>* Uncertainty rises near boundaries or sparse data

>* Use uncertainty when mistakes are costly
>* Combine predictions with confidence for decisions

>* Uncertainty reflects kernels, data, and approximations
>* Use maps to guide cautious decisions



### **3.3. Scaling Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_14/Lecture_A/image_03_03.jpg?v=1783966272" width="250">



>* Uncertainty estimates come with high computation
>* Large classification datasets often need approximations

>* Match GPC to dataset size and structure
>* Use approximations without losing useful uncertainty

>* Uncertainty maps show confidence and data gaps
>* Approximations affect trust in predictions



# <font color="#418FDE" size="6.5" uppercase>**Gaussian Processes**</font>


In this lecture, you learned to:
- Fit Gaussian Process Regression and interpret predictive mean and uncertainty. 
- Configure and compose Gaussian process kernels for practical modeling. 
- Apply Gaussian Process Classification while recognizing computational limits. 

In the next Lecture (Lecture B), we will go over 'Cross Decomposition'